In [3]:
import copy

def evaluate(board):
    score = 0
    values = {'p':1,'P':-1,'.':0}
    for row in board:
        for cell in row:
            score += values[cell]
    return score

def generate_moves(board, turn):
    moves = []
    for i in range(8):
        for j in range(8):
            piece = board[i][j]
            if piece == 'P' and turn == 'w':
                if i > 0 and board[i-1][j] == '.':
                    new_board = copy.deepcopy(board)
                    new_board[i-1][j] = 'P'
                    new_board[i][j] = '.'
                    moves.append(new_board)
            if piece == 'p' and turn == 'b':
                if i < 7 and board[i+1][j] == '.':
                    new_board = copy.deepcopy(board)
                    new_board[i+1][j] = 'p'
                    new_board[i][j] = '.'
                    moves.append(new_board)
    return moves

def beam_search(board, width, depth, turn):
    states = [(board, [], evaluate(board))]
    for d in range(depth):
        new_states = []
        for b, path, score in states:
            next_moves = generate_moves(b, turn)
            for nb in next_moves:
                new_path = path + [nb]
                new_score = evaluate(nb)
                new_states.append((nb, new_path, new_score))
        new_states.sort(key=lambda x:x[2], reverse=True)
        states = new_states[:width]
        if turn == 'w':
            turn = 'b'
        else:
            turn = 'w'
    best = states[0]
    return best[1], best[2]

board = [
['.','.','.','.','.','.','.','.'],
['P','P','P','P','P','P','P','P'],
['.','.','.','.','.','.','.','.'],
['.','.','.','.','.','.','.','.'],
['.','.','.','.','.','.','.','.'],
['.','.','.','.','.','.','.','.'],
['p','p','p','p','p','p','p','p'],
['.','.','.','.','.','.','.','.']
]

beam_width = 2
depth_limit = 2

moves, score = beam_search(board, beam_width, depth_limit, 'w')

print("best move sequence:")
for b in moves:
    for row in b:
        print(' '.join(row))
    print()
print("evaluation score:", score)

best move sequence:
P . . . . . . .
. P P P P P P P
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
p p p p p p p p
. . . . . . . .

P . . . . . . .
. P P P P P P P
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. p p p p p p p
p . . . . . . .

evaluation score: 0


In [4]:
import random
import math

def distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def total_distance(route):
    d = 0
    for i in range(len(route)-1):
        d += distance(route[i], route[i+1])
    return d

def hill_climb(points, iterations):
    current_route = points[:]
    current_distance = total_distance(current_route)
    
    for _ in range(iterations):
        i = random.randint(0, len(points)-1)
        j = random.randint(0, len(points)-1)
        if i != j:
            new_route = current_route[:]
            new_route[i], new_route[j] = new_route[j], new_route[i]
            new_distance = total_distance(new_route)
            if new_distance < current_distance:
                current_route = new_route
                current_distance = new_distance
                
    return current_route, current_distance

points = [(0,0),(2,3),(5,2),(6,6),(8,3)]
iterations = 1000

route, dist = hill_climb(points, iterations)

print("optimized route:")
for p in route:
    print(p)
print("total distance:", dist)

optimized route:
(0, 0)
(2, 3)
(5, 2)
(8, 3)
(6, 6)
total distance: 13.535657871264737


In [5]:
import random
import math

def dist(a,b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def total(r):
    d = 0
    for i in range(len(r)-1):
        d += dist(r[i], r[i+1])
    d += dist(r[-1], r[0])
    return d

def pop(cities, size):
    p = []
    for _ in range(size):
        r = cities[:]
        random.shuffle(r)
        p.append(r)
    return p

def select(p):
    p.sort(key=lambda x: total(x))
    return p[:len(p)//2]

def cross(p1,p2):
    n = len(p1)
    s = random.randint(0,n-2)
    e = random.randint(s+1,n-1)
    c = [None]*n
    for i in range(s,e+1):
        c[i] = p1[i]
    pos = 0
    for city in p2:
        if city not in c:
            while c[pos] != None:
                pos += 1
            c[pos] = city
    return c

def mutate(r, rate):
    for i in range(len(r)):
        if random.random() < rate:
            j = random.randint(0,len(r)-1)
            r[i], r[j] = r[j], r[i]

def ga(cities, pop_size, gens, rate):
    p = pop(cities, pop_size)
    for _ in range(gens):
        p = select(p)
        new_p = []
        while len(new_p) < pop_size:
            p1 = random.choice(p)
            p2 = random.choice(p)
            c = cross(p1,p2)
            mutate(c, rate)
            new_p.append(c)
        p = new_p
    best = min(p, key=lambda x: total(x))
    return best, total(best)

cities = [(0,0),(1,5),(2,3),(5,2),(6,6),(8,3),(7,7),(3,8),(4,4),(9,1)]

best, d = ga(cities, 6, 50, 0.1)

print("best route:")
for c in best:
    print(c)
print("total distance:", d)

best route:
(8, 3)
(9, 1)
(3, 8)
(7, 7)
(6, 6)
(0, 0)
(2, 3)
(5, 2)
(1, 5)
(4, 4)
total distance: 44.53142521844041


In [6]:
import copy

def max_load(allocation):
    loads = [sum(task[0] for task in proc) for proc in allocation]
    return max(loads)

def beam_search(tasks, procs, width, depth):
    states = [([[] for _ in range(procs)], [], 0)]
    
    for d in range(depth):
        new_states = []
        for alloc, path, _ in states:
            for i, task in enumerate(tasks):
                if task not in path:
                    for p in range(procs):
                        new_alloc = copy.deepcopy(alloc)
                        new_alloc[p].append(task)
                        new_path = path + [task]
                        score = max_load(new_alloc)
                        new_states.append((new_alloc, new_path, score))
        new_states.sort(key=lambda x:x[2])
        states = new_states[:width]
    best = states[0]
    return best[0], best[2]

tasks = [(5,1),(3,2),(2,3),(6,1),(4,2)]  
processors = 3
beam_width = 3
depth_limit = len(tasks)

alloc, score = beam_search(tasks, processors, beam_width, depth_limit)

print("optimized allocation:")
for i, proc in enumerate(alloc):
    print("processor", i+1, ":", proc)
print("maximum load:", score)

optimized allocation:
processor 1 : [(2, 3), (5, 1)]
processor 2 : [(3, 2), (6, 1)]
processor 3 : [(4, 2)]
maximum load: 9
